In [ ]:
import dotenv
import os
from sqlalchemy import create_engine, inspect
import pandas as pd
import boto3
from io import BytesIO

dotenv.load_dotenv()

In [ ]:
# create a authorized session using boto3 + credentials
# NOTE: ENSURE you have created a .env file in this project as below:
# AWS_ACCESS_KEY_ID=...
# AWS_SECRET_ACCESS_KEY=...
# POSTGRESQL_USERNAME=dataengineer
# POSTGRESQL_PASSWORD=...
session = boto3.Session(
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY"),
)
# create s3 client to interact with AWS S3
s3 = session.client(service_name='s3')

s3_bucket = '<YOUR S3 BUCKET>'
s3_path_prefix = "olist"

In [ ]:
host = 'cm-de-k1-db.c9ms60q6cw37.ap-southeast-2.rds.amazonaws.com'
port = '5432'
database = 'postgres'
user = os.environ.get("POSTGRESQL_USERNAME")
password = os.environ.get("POSTGRESQL_PASSWORD")

# Construct the connection string
# The 'postgresql+psycopg2' part specifies the dialect and driver
connection_string = f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}'

# Create the SQLAlchemy engine
olist_engine = create_engine(connection_string)
olist_conn = olist_engine.connect()

In [ ]:
inspector = inspect(olist_engine)
table_names = inspector.get_table_names(schema='olist')

for table in table_names:
    print(f"Ingesting '{table}' . . . ")
    s3_key = f"{s3_path_prefix}/{table}/{table}.parquet"
    df = pd.read_sql_table(table_name=table, con=olist_conn, schema='olist')
    out_buffer = BytesIO()
    df.to_parquet(out_buffer, index=False, engine='pyarrow')
    s3.put_object(Bucket=s3_bucket, Key=s3_key, Body=out_buffer.getvalue())